# 06 — Refit honesto + amplitude-forward no OD da estação EF01 (CETESB)

**Objetivo:** bater a régua do OD (sazonal-naive 0,1525 rolante / 0,1550 holdout) removendo a causa raiz dos fracassos anteriores: o treino congela em junho (amplitude pequena) e o holdout vive em julho (amplitude 0,77→1,83). Mesmo janelamento do 00b–05 (L=8640/H=288, split 70/15/15 + holdout de 10 dias, segmento limpo 01/06 → 21/07).
**Estágio A — refit honesto (comparável):** `lgbm_A` + `dlw_A` retreinados em `tr+va` (HP congelados do 05; `te` segue limpo p/ NNLS e tabela) · **fine-tune de escala do LSTNet** (`lstnet_ft`: backbone congelado, só `gamma/beta` do RevIN + matriz AR, 5 épocas LR 1e-4, pool antes do `te`) · **ridge do fator de escala** (linear explícito, 4 features) · **amp-trend** (prevê o escalar `amp(amanhã)` por tendência nos `ptp` diários e reescala o template).
**Estágio B — walk-forward (tabela separada "deploy realista"):** para cada origem do holdout, refit causal (`lgbm_B` + `dlw_B` fine-tunado de A + amp-trend); modelos diários efêmeros.
**Dados:** `dados/ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv` — ver `dados/README.md`.

In [1]:
import gc
import json
import pickle
import random
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110})

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
CSV = ROOT / "dados" / "ef01-mogi-das-cruzes_oxigenio-dissolvido_2026-06-01_a_2026-08-31.csv"
OUT = ROOT / "resultados" / "06-refit-od"
(OUT / "modelos").mkdir(parents=True, exist_ok=True)
(OUT / "figs").mkdir(parents=True, exist_ok=True)

# --- protocolo travado (igual ao 00b) ---
L, H = 8640, 288
SEASON = 288
INTERP_LIMIT = 24
SEG_FIM = "2026-07-21 01:05"
HOLDOUT_DIAS = 10
# --- 06: refit + amplitude-forward ---
LN = 2016
CTX = 2016
LGB_EST, LGB_LR, LGB_LEAVES = 150, 0.05, 31
LGB_STRIDE = 2
DL_FIX_EP = 2     # épocas fixas (sem val em tr+va)
TRAIN_STRIDE = 4      # stride do DLinear (05: tr+va)
FT_EP, FT_LR = 5, 1e-4   # fine-tune de escala do LSTNet: poucas épocas, LR baixo, sem val por construção
W_ALPHA = 3.0
JIT_LO, JIT_HI = 0.8, 1.25
WF_DIAS = 21      # walk-forward: pool causal dos últimos 21 dias antes de cada origem
TREND_J = 14      # tendência de amplitude nos últimos 14 ptps diários
ENS_STRIDE = 4
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
print("ROOT:", ROOT, "| CSV existe:", CSV.exists(), "| torch:", torch.__version__)

ROOT: /home/marcos/temporal-model | CSV existe: True | torch: 2.14.0+cpu


## 1. Carga
Formato CETESB: `;`, decimal com vírgula, `windows-1252`, linha 1 = validação, linha 2 = cabeçalho.

In [2]:
df = pd.read_csv(CSV, sep=";", decimal=",", encoding="windows-1252",
                 skiprows=1, parse_dates=["Data hora"], dayfirst=True, na_values=[""])
colvar = [c for c in df.columns if c != "Data hora"][0]
df = df.rename(columns={"Data hora": "ds", colvar: "y"}).sort_values("ds").reset_index(drop=True)
print(colvar, "|", df.shape, df["ds"].min(), "→", df["ds"].max())
print("faltantes:", int(df['y'].isna().sum()), f"({100*df['y'].isna().mean():.1f}%)")
df.describe()

Oxigênio Dissolvido (mg/L) | (26209, 2) 2026-06-01 00:00:00 → 2026-08-31 00:00:00
faltantes: 4767 (18.2%)


,ds,y
count,26209,21442.000000
mean,2026-07-16 12:00:00,6.668501
min,2026-06-01 00:00:00,4.840000
25%,2026-06-23 18:00:00,6.320000
50%,2026-07-16 12:00:00,6.630000
75%,2026-08-08 06:00:00,6.950000
max,2026-08-31 00:00:00,8.910000
std,NaN,0.575499


## 2. EDA — perfil, o gap de 16 dias e ciclo diário

In [3]:
isna = df["y"].isna().to_numpy()
bounds = np.where(np.diff(np.concatenate([[False], isna, [False]])))[0]
runs = sorted([(bounds[i], bounds[i+1]-1) for i in range(0, len(bounds), 2)],
              key=lambda r: r[1]-r[0], reverse=True)
print("top 5 gaps:")
for a, b in runs[:5]:
    print(f"  {df.ds[a]} → {df.ds[b]}  ({(b-a+1)*5/60:.1f} h)")

fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=False)
ax[0].plot(df["ds"], df["y"], lw=0.4)
ax[0].axvspan(pd.Timestamp("2026-07-21 01:10"), pd.Timestamp("2026-08-06 11:30"),
              color="r", alpha=0.2, label="sensor morto (16,4 dias)")
ax[0].set_title("OD EF01 — série completa (faixa vermelha = gap, fora do experimento)")
ax[0].set_ylabel("OD (mg/L)")
ax[0].legend(fontsize=8)
df["y"].hist(bins=60, ax=ax[1])
ax[1].set_title("Distribuição do OD")
df.assign(hora=df["ds"].dt.hour).boxplot(column="y", by="hora", ax=ax[2], grid=False)
ax[2].set_title("OD por hora do dia (ciclo diário?)")
ax[2].set_xlabel("hora")
fig.tight_layout()
fig.savefig(OUT / "figs" / "01-eda.png")
print("fig salva:", OUT / "figs" / "01-eda.png")

top 5 gaps:
  2026-07-21 01:10:00 → 2026-08-06 11:30:00  (394.4 h)
  2026-06-30 22:05:00 → 2026-06-30 23:55:00  (1.9 h)
  2026-07-11 10:15:00 → 2026-07-11 10:30:00  (0.3 h)
  2026-07-16 09:40:00 → 2026-07-16 09:45:00  (0.2 h)
  2026-07-20 20:10:00 → 2026-07-20 20:10:00  (0.1 h)


fig salva: /home/marcos/temporal-model/resultados/06-refit-od/figs/01-eda.png


## 3. Limpeza + recorte do segmento limpo
Grade de 5 min, interpolação máx. 2 h e **corte em 21/07 01:05** (antes do gap). Tudo a jusante usa só o segmento 01/06 → 21/07.

In [4]:
idx = pd.date_range(df["ds"].min(), df["ds"].max(), freq="5min")
s_full = df.set_index("ds")["y"].reindex(idx)
s = s_full.loc[:SEG_FIM].interpolate(method="time", limit=INTERP_LIMIT)
print(f"segmento: {s.index.min()} → {s.index.max()} ({len(s)} slots = {len(s)*5/60/24:.1f} dias)")
print(f"NaN após interpolação (limite {INTERP_LIMIT}): {int(s.isna().sum())}")

amostra = slice("2026-06-08", "2026-06-15")
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(s_full[amostra].index, s_full[amostra].values, ".", ms=2, label="cru (com faltantes)")
ax.plot(s[amostra].index, s[amostra].values, lw=0.8, label=f"interpolado (limite {INTERP_LIMIT})")
ax.legend(); ax.set_title("Exemplo de preenchimento — semana 08–15/06")
fig.tight_layout(); fig.savefig(OUT / "figs" / "02-limpeza.png")
print("fig salva")

segmento: 2026-06-01 00:00:00 → 2026-07-21 01:05:00 (14414 slots = 50.0 dias)
NaN após interpolação (limite 24): 0
fig salva


## 4. Estacionariedade (ADF) e decomposição STL
Idêntico ao 00b (últimos 4032 pontos do treino, período 288).

In [5]:
n_total = len(s)
n_train = int(n_total * 0.70)
train = s.iloc[:n_train].dropna()
stat, pval, *_ = adfuller(train.values)
print(f"ADF stat={stat:.2f} p-valor={pval:.3g} → {'estacionária' if pval < 0.05 else 'NÃO estacionária'}")

stl = STL(train.iloc[-4032:], period=SEASON, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 6)
fig.savefig(OUT / "figs" / "03-stl.png")
print("fig salva")

ADF stat=-4.93 p-valor=3.03e-05 → estacionária


fig salva


## 5. Janelamento + holdout puro + série diária de amplitude
Janelas `(L=8640 → H=288)` idênticas ao 00b (split 70/15/15 sem shuffle + holdout de 10 dias + 10 origens diárias). A série diária (`ptp`/`mean` por dia) alimenta o amp-trend — só dias 100% observados.

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

v = s.to_numpy()
W = sliding_window_view(v, L + H)
ok = ~np.isnan(W).any(axis=1)
W = W[ok]
X, Y = W[:, :L], W[:, L:]
ends = s.index[L + H - 1:][ok]
n = len(X)
ZONE = s.index.max() - pd.Timedelta(days=HOLDOUT_DIAS)
is_hold = ends >= (ZONE + pd.Timedelta(minutes=5 * (H - 1)))
ho = np.where(is_hold)[0]
pre = np.where(~is_hold)[0]
i1, i2 = int(len(pre) * 0.70), int(len(pre) * 0.85)
tr, va, te = pre[:i1], pre[i1:i2], pre[i2:]
trva = np.concatenate([tr, va])
for k, idx in {"train": tr, "val": va, "test": te, "holdout": ho}.items():
    print(f"{k}: {len(idx)} janelas | alvos {ends[idx[0]].date()} → {ends[idx[-1]].date()}")
print(f"janelas descartadas (com NaN): {len(s) - L - H + 1 - n}")
daily_ends = [ZONE + pd.Timedelta(minutes=5 * (H - 1 + H * k)) for k in range(HOLDOUT_DIAS)]
daily_idx = np.array([int(np.where(ends == d)[0][0]) for d in daily_ends])
print("dias previstos:", [str(ends[i].date()) for i in daily_idx])
TE_START = ends[te[0]]

# série diária causal (dias completos, sem NaN)
daily = s.to_frame("y").resample("1D").agg(ptp=("y", lambda a: float(a.max() - a.min())),
                                            mean=("y", "mean"), n=("y", "size"))
daily = daily[daily["n"] == 288].drop(columns="n")
print(f"dias completos: {len(daily)} ({daily.index.min().date()} → {daily.index.max().date()})")
print("ptp dos últimos 12 dias:", daily["ptp"].iloc[-12:].round(2).tolist())

train: 2025 janelas | alvos 2026-07-01 → 2026-07-09
val: 434 janelas | alvos 2026-07-09 → 2026-07-10
test: 434 janelas | alvos 2026-07-10 → 2026-07-12
holdout: 2594 janelas | alvos 2026-07-12 → 2026-07-21
janelas descartadas (com NaN): 0
dias previstos: ['2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16', '2026-07-17', '2026-07-18', '2026-07-19', '2026-07-20', '2026-07-21']
dias completos: 50 (2026-06-01 → 2026-07-20)
ptp dos últimos 12 dias: [0.7, 0.64, 0.77, 0.79, 1.17, 0.97, 1.11, 1.25, 1.42, 1.57, 1.83, 1.82]


## 6. Baselines baratos + sazonal reescalado
Persistência, sazonal-naive (lag 288) e média móvel 288 (iguais ao 00b) + `saz_escalado` do 05 (template recentrado, desvios × `std(ctx)/std(ref)`, clip [0,5, 2,0] — analítico, sem treino).

In [7]:
def mae(a, b): return float(mean_absolute_error(a.ravel(), b.ravel()))
def rmse(a, b): return float(np.sqrt(mean_squared_error(a.ravel(), b.ravel())))
def mape(a, b, eps=1e-6): return float(np.mean(np.abs((a - b) / np.maximum(np.abs(a), eps))) * 100)
def smape(a, b, eps=1e-6): return float(np.mean(2*np.abs(a-b) / (np.abs(a)+np.abs(b)+eps)) * 100)

def metricas(y_true, y_pred):
    return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred),
            "MAPE": mape(y_true, y_pred), "sMAPE": smape(y_true, y_pred)}

def cheap_preds(X_):
    return {
        "persistencia": np.repeat(X_[:, -1:], H, axis=1),
        "sazonal_naive_288": np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1),
        "media_movel_288": np.repeat(X_[:, -SEASON:].mean(axis=1, keepdims=True), H, axis=1),
    }

def snaive(X_):
    return np.stack([X_[:, L - SEASON + h] for h in range(H)], axis=1)

def saz_escalado(X_):
    S = snaive(X_)
    Ctx = X_[:, -CTX:]
    Ref = X_[:, L - SEASON - CTX:L - SEASON]
    mu_c = Ctx.mean(axis=1, keepdims=True)
    mu_r = Ref.mean(axis=1, keepdims=True)
    sc = Ctx.std(axis=1, keepdims=True) / np.maximum(Ref.std(axis=1, keepdims=True), 1e-6)
    sc = np.clip(sc, 0.5, 2.0)
    return (mu_c + (S - mu_r) * sc).astype(np.float64)

Xte, Yte = X[te], Y[te]
pred_te = cheap_preds(Xte)
pred_te["saz_escalado"] = saz_escalado(Xte)
print("teste rolante (baratos):")
print(pd.DataFrame({m: metricas(Yte, p) for m, p in pred_te.items()}).T.round(4).to_string())

teste rolante (baratos):
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.2371  0.3019  3.3575  3.3530
sazonal_naive_288  0.1525  0.1770  2.1668  2.1918
media_movel_288    0.1900  0.2468  2.6526  2.6942
saz_escalado       0.1506  0.1677  2.1472  2.1577


## 7. Estágio A — LGBM-multiplicativo em `tr+va` + ridge do fator de escala
`lgbm_A`: mesmos HP/features/peso de recência do 05, agora em `tr+va` (ganha os dias grandes do início de julho); `te` segue limpo. Salvo em `modelos/lgbm_mult_A_steps.pkl` (gitignore). `ridge_scale`: alvo = escala oráculo por origem (`s* = <Y−mu_c, S−mu_r>/<S−mu_r, S−mu_r>`, clip [0,5, 2,0]) a partir das 4 features de amplitude — se o GBM ignora essas features, o linear pode extrapolar melhor.

In [8]:
import lightgbm as lgb
from sklearn.linear_model import Ridge

def base_feats(Xb, E):
    cols = [Xb[:, -k] for k in [1, 2, 3, 6, 12, 24, 36, 72, 144, 287, 288, 289, 576, 2016]]
    phase = np.stack([Xb[:, L - 288*k] for k in range(1, 8)], axis=1)
    cols += [phase.mean(1), phase.std(1)]
    for w in [12, 36, 144, 288]:
        cols += [Xb[:, -w:].mean(1), Xb[:, -w:].std(1)]
    cols += [Xb[:, -2016:].mean(1)]
    F = np.stack(cols, axis=1)
    em = (E.hour.to_numpy()*60 + E.minute.to_numpy()).astype(np.float32)
    return F.astype(np.float32), em, phase

def amp_feats(Xb, phase):
    Ctx = Xb[:, -CTX:]
    Ref = Xb[:, L - SEASON - CTX:L - SEASON]
    ctx_std = Ctx.std(axis=1)
    ctx_ptp = Ctx.max(axis=1) - Ctx.min(axis=1)
    seas_ptp = phase.max(axis=1) - phase.min(axis=1)
    ratio = ctx_std / np.maximum(Ref.std(axis=1), 1e-6)
    return np.stack([ctx_std, ctx_ptp, seas_ptp, ratio], axis=1).astype(np.float32)

def hour_sincos(em, j):
    hh = ((em - (H - 1 - j)*5) % 1440 // 60).astype(np.float32)
    return np.sin(2*np.pi*hh/24).astype(np.float32), np.cos(2*np.pi*hh/24).astype(np.float32)

trva2 = trva[::LGB_STRIDE]
Xb_tr = X[trva2]  # hoist (OOM se dentro da compreensão)
Ftr, emtr, Phtr = base_feats(Xb_tr, ends[trva2])
Atr = amp_feats(Xb_tr, Phtr)
Ftr = np.column_stack([Ftr, Atr])
Str = snaive(Xb_tr)
Qtr = np.clip((Y[trva2] / np.maximum(Str, 1e-6)).astype(np.float32), 0.5, 1.5)
w_tr = (1.0 + W_ALPHA * np.linspace(0, 1, len(trva2))).astype(np.float32)
print(f"A-refit: features {Ftr.shape} | {len(trva2)} janelas (tr+va) | razão std: {Qtr.std():.4f}")

models_A = []
t0 = time.time()
for j in range(H):
    sh, ch = hour_sincos(emtr, j)
    m = lgb.LGBMRegressor(n_estimators=LGB_EST, learning_rate=LGB_LR, num_leaves=LGB_LEAVES,
                          verbosity=-1, force_col_wise=True)
    m.fit(np.column_stack([Ftr, sh, ch]), Qtr[:, j], sample_weight=w_tr)
    models_A.append(m)
    if (j + 1) % 72 == 0:
        print(f"  lgbm_A {j+1}/{H} ...", flush=True)
print(f"lgbm_A: {len(models_A)} modelos em {time.time()-t0:.0f}s")
with open(OUT / "modelos" / "lgbm_mult_A_steps.pkl", "wb") as f:
    pickle.dump(models_A, f)

def prevê_lgbm(idxs, models):
    ii = np.asarray(idxs)
    Xb = X[ii]  # hoist
    F, em, Ph = base_feats(Xb, ends[ii])
    F = np.column_stack([F, amp_feats(Xb, Ph)])
    S = snaive(Xb)
    P = np.empty((len(ii), H), dtype=np.float32)
    for j, m in enumerate(models):
        sh, ch = hour_sincos(em, j)
        P[:, j] = S[:, j] * np.clip(m.predict(np.column_stack([F, sh, ch])), 0.5, 1.5)
    return P

imp = np.mean([m.booster_.feature_importance(importance_type="gain") for m in models_A], axis=0)
nomes = ["lag1", "lag2", "lag3", "lag6", "lag12", "lag24", "lag36", "lag72", "lag144",
         "lag287", "lag288", "lag289", "lag576", "lag2016", "seasmean7", "seasstd7",
         "rm12", "rs12", "rm36", "rs36", "rm144", "rs144", "rm288", "rs288", "rm2016",
         "ctx_std", "ctx_ptp", "seas_ptp7", "amp_ratio",
         "hora_sin", "hora_cos"]
ordem = np.argsort(imp)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh([nomes[k] for k in ordem], imp[ordem])
ax.set_title("LightGBM-A (tr+va) — importância média (gain, 288 modelos)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "07-importancia-lgbm.png")
print("top features:", [(nomes[k], round(float(imp[k]), 1)) for k in ordem[:5]])

# ridge do fator de escala (só 4 features de amplitude)
S_trva, Y_trva = snaive(X[trva2]), Y[trva2]
mu_c = X[trva2][:, -CTX:].mean(axis=1, keepdims=True)
mu_r = X[trva2][:, L - SEASON - CTX:L - SEASON].mean(axis=1, keepdims=True)
a, b = Y_trva - mu_c, S_trva - mu_r
oracle = np.clip((a * b).sum(axis=1) / np.maximum((b * b).sum(axis=1), 1e-6), 0.5, 2.0)
ridge = Ridge(alpha=1.0).fit(Atr, oracle)
print(f"ridge: R² treino={ridge.score(Atr, oracle):.3f} | coef={ridge.coef_.round(3).tolist()}")

def prevê_ridge(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]
    _, _, Ph = base_feats(Xb, ends[ii])
    sc = np.clip(ridge.predict(amp_feats(Xb, Ph)), 0.5, 2.0)[:, None]
    S = snaive(Xb)
    mu_c = Xb[:, -CTX:].mean(axis=1, keepdims=True)
    mu_r = Xb[:, L - SEASON - CTX:L - SEASON].mean(axis=1, keepdims=True)
    return (mu_c + (S - mu_r) * sc).astype(np.float64)

A-refit: features (1230, 29) | 1230 janelas (tr+va) | razão std: 0.0254


  lgbm_A 72/288 ...


  lgbm_A 144/288 ...


  lgbm_A 216/288 ...


  lgbm_A 288/288 ...


lgbm_A: 288 modelos em 16s


top features: [('rm2016', 1.8), ('rm288', 1.2), ('lag144', 1.1), ('rs288', 1.0), ('lag289', 1.0)]
ridge: R² treino=0.116 | coef=[-0.5720000267028809, -0.19200000166893005, -0.14499999582767487, 0.5460000038146973]


## 8. Estágio A — DLinear refit + fine-tune de escala do LSTNet + amplitude-forward
**dlw_A:** mesma arquitetura/loss ponderada/jitter do 05, `DL_FIX_EP=2` épocas em `tr+va` (o 05 convergiu na ep 1; sem val disponível, margem de +1 documentada). **lstnet_ft:** checkpoint do 02b, backbone congelado, só `gamma/beta` + `ar` ajustados (5 épocas, LR 1e-4) em janelas **anteriores ao `te`** — `te` segue 100% limpo para todos. **amp_trend:** para cada origem, `amp_hat(D)` por tendência linear nos últimos 14 `ptp` diários causais; `pred = mu_ctx + (S − mu_src)·amp_hat/ptp_src` (clip [0,3, 3,0]).

In [9]:
class DLinearLite(nn.Module):
    def __init__(self, k=25):
        super().__init__()
        self.pool = nn.AvgPool1d(k, stride=1, padding=k // 2)
        self.lin_t = nn.Linear(LN, H)
        self.lin_s = nn.Linear(LN, H)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        mu = x.mean(dim=1, keepdim=True); sg = x.std(dim=1, keepdim=True).clamp_min(1e-3)
        xn = self.gamma * (x - mu) / sg + self.beta
        t = self.pool(xn.unsqueeze(1)).squeeze(1)
        y = self.lin_t(t) + self.lin_s(xn - t)
        return (y - self.beta) / self.gamma.clamp_min(1e-3) * sg

def monta_res(idxs):
    ii = np.asarray(idxs)
    return X[ii][:, -LN:].astype(np.float32), (Y[ii] - snaive(X[ii])).astype(np.float32)

Xr, Rr = monta_res(trva[::TRAIN_STRIDE])
w_dl = (1.0 + W_ALPHA * np.linspace(0, 1, len(Xr))).astype(np.float32)
print(f"dlw_A: treino {Xr.shape} (tr+va, stride 4), {DL_FIX_EP} épocas fixas")
dlw_A = DLinearLite().to(DEVICE)
opt = torch.optim.Adam(dlw_A.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(torch.from_numpy(Xr), torch.from_numpy(Rr),
                                  torch.from_numpy(w_dl)), batch_size=512, shuffle=True)
rng = np.random.default_rng(SEED)
t0 = time.time()
for ep in range(1, DL_FIX_EP + 1):
    dlw_A.train()
    for xb, yb, wb in loader:
        mu = xb.mean(dim=1, keepdim=True)
        f = torch.from_numpy(rng.uniform(JIT_LO, JIT_HI, size=(len(xb), 1)).astype(np.float32))
        opt.zero_grad()
        se = ((dlw_A(mu + f * (xb - mu)) - f * yb) ** 2).mean(dim=1)
        (se * wb / wb.mean()).mean().backward(); opt.step()
    print(f"dlw_A ep {ep:02d} ({time.time()-t0:.0f}s)", flush=True)
torch.save({"state": dlw_A.state_dict()}, OUT / "modelos" / "dlinear_A_od.pt")
dlw_A.eval()

@torch.no_grad()
def prevê_dlw(idxs, model, batch=256):
    ii = np.asarray(idxs)
    Ps = snaive(X[ii])
    outs = []
    Xt = torch.from_numpy(X[ii][:, -2016:].astype(np.float32))
    for b in range(0, len(Xt), batch):
        outs.append(model(Xt[b:b+batch]).numpy())
    return Ps + np.concatenate(outs)

# --- LSTNet: régua 02b + fine-tune de escala ---
class LSTNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv1d(3, 32, kernel_size=12, stride=6)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.skipcell = nn.GRUCell(32, 32)
        self.head = nn.Linear(96, 288)
        self.ar = nn.Linear(288, 288)
        self.drop = nn.Dropout(0.1)
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))
    def forward(self, xv, tod):
        mu = xv.mean(dim=1, keepdim=True); sg = xv.std(dim=1, keepdim=True).clamp_min(1e-3)
        vn = self.gamma * (xv - mu) / sg + self.beta
        f = self.drop(torch.relu(self.conv(torch.cat([vn.unsqueeze(1), tod.transpose(1, 2)], dim=1))))
        f = f.transpose(1, 2)
        _, h = self.gru(f)
        B, T, _ = f.shape
        hs = torch.zeros(B, 32, device=f.device)
        states = [hs]
        for t in range(T):
            prev = states[t - 48] if t - 48 >= 0 else states[0]
            hs = self.skipcell(f[:, t, :], prev)
            states.append(hs)
        g = self.gamma.clamp_min(1e-3)
        yn = self.head(self.drop(torch.cat([h.squeeze(0), hs], dim=1)))
        ya = self.ar(vn[:, -288:])
        return (yn + ya - self.beta) / g * sg + mu

ckpt02 = torch.load(ROOT / "resultados" / "02b-lstnet-od" / "modelos" / "lstnet_od.pt",
                    map_location="cpu", weights_only=False)
SIN5 = np.sin(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
COS5 = np.cos(2*np.pi*(s.index.hour.to_numpy()*60+s.index.minute.to_numpy())/1440.0).astype(np.float32)
Tln = sliding_window_view(np.stack([SIN5, COS5], axis=1), 2016, axis=0).transpose(0, 2, 1).astype(np.float32)
pos_end = s.index.get_indexer(ends - pd.Timedelta(minutes=5*H))
rowln = pos_end - 2016 + 1
Wln2 = sliding_window_view(s.to_numpy().astype(np.float32), 2016)

@torch.no_grad()
def prevê_lstnet(idxs, state, batch=128):
    net = LSTNet1D().to(DEVICE)
    net.load_state_dict(state); net.eval()
    ii = np.asarray(idxs)
    outs = []
    for b in range(0, len(ii), batch):
        xb = torch.from_numpy(Wln2[rowln[ii[b:b+batch]]])
        tb = torch.from_numpy(Tln[rowln[ii[b:b+batch]]])
        outs.append(net(xb, tb).numpy())
    del net
    return np.concatenate(outs)

# fine-tune: só escala (gamma/beta + ar), pool estrito antes do te, épocas fixas
ft_pool = pre[ends[pre] < TE_START][::2]
print(f"fine-tune escala: {len(ft_pool)} janelas (fins {ends[ft_pool[0]].date()} → {ends[ft_pool[-1]].date()}), {FT_EP} eps LR={FT_LR}")
ft_net = LSTNet1D().to(DEVICE)
ft_net.load_state_dict(ckpt02["state"])
for p in ft_net.parameters():
    p.requires_grad = False
for name in ["gamma", "beta"]:
    getattr(ft_net, name).requires_grad = True
ft_net.ar.weight.requires_grad = True
ft_net.ar.bias.requires_grad = True
ft_opt = torch.optim.Adam([p for p in ft_net.parameters() if p.requires_grad], lr=FT_LR)
ft_loader = DataLoader(TensorDataset(torch.from_numpy(X[ft_pool][:, -2016:].astype(np.float32)),
                                     torch.from_numpy(Y[ft_pool].astype(np.float32))),
                       batch_size=128, shuffle=True)
Xtod_full = torch.from_numpy(Tln[rowln[ft_pool]])
ft_net.train()
t0 = time.time()
for ep in range(1, FT_EP + 1):
    for bi, (xb, yb) in enumerate(ft_loader):
        n0 = bi * 128
        tb = Xtod_full[n0:n0 + len(xb)]
        ft_opt.zero_grad()
        (((ft_net(xb, tb) - yb) ** 2).mean()).backward(); ft_opt.step()
    print(f"ft ep {ep:02d} ({time.time()-t0:.0f}s)", flush=True)
ft_state = {k: v.detach().cpu() for k, v in ft_net.state_dict().items()}
torch.save({"state": ft_state}, OUT / "modelos" / "lstnet_ft_od.pt")
del ft_net, ft_loader, Xtod_full; gc.collect()
print("lstnet_ft salvo")

# --- amplitude-forward: tendência nos ptp diários causais ---
dptp = daily["ptp"].to_numpy()
ddays = daily.index
def amp_hat_para(dia_alvo):
    j = int(np.searchsorted(ddays.values, np.datetime64(dia_alvo)))
    hist = dptp[max(0, j - TREND_J):j]
    if len(hist) < 3:
        return float(hist[-1]) if len(hist) else 1.0
    a, b = np.polyfit(np.arange(len(hist)), hist, 1)
    return float(max(0.2, a * len(hist) + b))

def prevê_amptrend(idxs):
    ii = np.asarray(idxs)
    Xb = X[ii]
    S = snaive(Xb)
    mu_c = Xb[:, -CTX:].mean(axis=1, keepdims=True)
    out = np.empty_like(S)
    for r, e in enumerate(ends[ii]):
        dia = (e + pd.Timedelta(minutes=5)).date()  # dia-alvo (primeiro slot do horizonte)
        src = s.loc[str(dia - pd.Timedelta(days=1))]
        mu_s, ptp_s = float(src.mean()), float(src.max() - src.min())
        f = np.clip(amp_hat_para(dia) / max(ptp_s, 1e-6), 0.3, 3.0)
        out[r] = mu_c[r] + (S[r] - mu_s) * f
    return out

dlw_A: treino (615, 2016) (tr+va, stride 4), 2 épocas fixas


dlw_A ep 01 (0s)


dlw_A ep 02 (0s)


fine-tune escala: 1230 janelas (fins 2026-07-01 → 2026-07-10), 5 eps LR=0.0001


ft ep 01 (2s)


ft ep 02 (4s)


ft ep 03 (5s)


ft ep 04 (7s)


ft ep 05 (9s)


lstnet_ft salvo


## 9. Estágio A — inferência + ensemble NNLS no `te`
Todas as origens de teste/holdout com os modelos do Estágio A (+ régua 02b congelada). Pesos NNLS em 8 colunas, fitados só no `te` (limpo para todos).

In [10]:
from scipy.optimize import nnls

t0 = time.time()
Ps_te = snaive(Xte)
Se_te = pred_te["saz_escalado"]
Pn02_te = prevê_lstnet(te, ckpt02["state"])
PnFT_te = prevê_lstnet(te, ft_state)
GmA_te = prevê_lgbm(te, models_A)
DwA_te = prevê_dlw(te, dlw_A)
RgA_te = prevê_ridge(te)
AtA_te = prevê_amptrend(te)
print(f"inferência teste em {time.time()-t0:.0f}s")

va2 = te[::ENS_STRIDE]  # NNLS no te (limpo); stride p/ custo do amptrend
cols = [snaive(X[va2]), saz_escalado(X[va2]), prevê_lstnet(va2, ckpt02["state"]),
        prevê_lstnet(va2, ft_state), prevê_lgbm(va2, models_A), prevê_dlw(va2, dlw_A),
        prevê_ridge(va2), prevê_amptrend(va2)]
A = np.column_stack([c.ravel() for c in cols])
wA, _ = nnls(A, Y[va2].ravel())
nomesA = ["sazonal", "saz_esc", "lstnet02", "lstnet_ft", "lgbm_A", "dlw_A", "ridge", "amptrend"]
pesosA = {k: round(float(v), 4) for k, v in zip(nomesA, wA)}
json.dump({"pesos": pesosA, "mode": "nnls-ensemble Estágio A (fit no te)"},
          open(OUT / "modelos" / "ensemble.json", "w"))
json.dump({"mode": "refit tr+va + finetune-escala + ridge + amptrend + nnls(te)", "LN": LN, "CTX": CTX},
          open(OUT / "modelos" / "normalizacao.json", "w"))
print("pesos ensemble A (nnls no te):", pesosA)

def ensemble_A(Ps, Se, Pn02, PnFT, Gm, Dw, Rg, At):
    return wA[0]*Ps + wA[1]*Se + wA[2]*Pn02 + wA[3]*PnFT + wA[4]*Gm + wA[5]*Dw + wA[6]*Rg + wA[7]*At

EnA_te = ensemble_A(Ps_te, Se_te, Pn02_te, PnFT_te, GmA_te, DwA_te, RgA_te, AtA_te)
for m, p in {"lstnet02": Pn02_te, "lstnet_ft": PnFT_te, "lgbm_A": GmA_te,
              "dlw_A": DwA_te, "ridge": RgA_te, "amptrend": AtA_te, "ensA": EnA_te}.items():
    print(f"{m} teste: MAE={mae(Yte, p):.4f} RMSE={rmse(Yte, p):.4f}", flush=True)

inferência teste em 1s


pesos ensemble A (nnls no te): {'sazonal': 0.0, 'saz_esc': 0.0, 'lstnet02': 0.5681, 'lstnet_ft': 0.3168, 'lgbm_A': 0.0, 'dlw_A': 0.0, 'ridge': 0.0, 'amptrend': 0.1234}
lstnet02 teste: MAE=0.0981 RMSE=0.1198


lstnet_ft teste: MAE=0.1097 RMSE=0.1285


lgbm_A teste: MAE=0.1662 RMSE=0.2037


dlw_A teste: MAE=0.1850 RMSE=0.2153


ridge teste: MAE=0.1450 RMSE=0.1641


amptrend teste: MAE=0.3046 RMSE=0.3406


ensA teste: MAE=0.0920 RMSE=0.1109


## 10. Estágio B — walk-forward diário no holdout (deploy realista)
Para cada um dos 10 dias: pool causal dos últimos 21 dias (`ends` ≤ início do dia-alvo), `lgbm_B` fresh (mesmos HP) + `dlw_B` (3 épocas LR 3e-4 a partir de A) + amp-trend. Modelos diários efêmeros (só métricas persistem). Ensemble com os pesos de A renormalizados no subconjunto disponível — limitação documentada.

In [11]:
DL_B_EP, DL_B_LR = 3, 3e-4
sub = [0, 1, 3, 4, 5, 7]  # sazonal, saz_esc, lstnet_ft, lgbm, dlw, amptrend
wB = wA[sub] / wA[sub].sum()
print("pesos B (subset A renormalizado):",
      {k: round(float(v), 4) for k, v in zip(["sazonal", "saz_esc", "lstnet_ft", "lgbm_B", "dlw_B", "amptrend"], wB)})

resB, t0 = [], time.time()
for k, di in enumerate(daily_idx):
    E = ends[di]
    dia = (E + pd.Timedelta(minutes=5)).date()
    cut = E - pd.Timedelta(minutes=5 * H)
    pool = np.where((ends <= cut) & (ends > cut - pd.Timedelta(days=WF_DIAS)))[0]
    Xd = X[[di]]
    Ps = snaive(Xd); Se = saz_escalado(Xd); PnFT = prevê_lstnet([di], ft_state)
    At = prevê_amptrend([di])
    # lgbm_B fresh no pool
    pb = pool[::LGB_STRIDE]
    Xb = X[pb]
    Fb, emb, Phb = base_feats(Xb, ends[pb])
    Fb = np.column_stack([Fb, amp_feats(Xb, Phb)])
    Sb = snaive(Xb)
    Qb = np.clip((Y[pb] / np.maximum(Sb, 1e-6)).astype(np.float32), 0.5, 1.5)
    wb = (1.0 + W_ALPHA * np.linspace(0, 1, len(pb))).astype(np.float32)
    mods = []
    for j in range(H):
        sh, ch = hour_sincos(emb, j)
        m = lgb.LGBMRegressor(n_estimators=LGB_EST, learning_rate=LGB_LR, num_leaves=LGB_LEAVES,
                              verbosity=-1, force_col_wise=True)
        m.fit(np.column_stack([Fb, sh, ch]), Qb[:, j], sample_weight=wb)
        mods.append(m)
    Fd, emd, Phd = base_feats(Xd, ends[[di]])
    Fd = np.column_stack([Fd, amp_feats(Xd, Phd)])
    Gm = np.empty((1, H), dtype=np.float32)
    for j, m in enumerate(mods):
        sh, ch = hour_sincos(emd, j)
        Gm[:, j] = Ps[:, j] * np.clip(m.predict(np.column_stack([Fd, sh, ch])), 0.5, 1.5)
    del mods, Fb, Qb; gc.collect()
    # dlw_B: 3 épocas a partir de A no pool
    Xp, Rp = monta_res(pool[::TRAIN_STRIDE])
    netB = DLinearLite().to(DEVICE)
    netB.load_state_dict(dlw_A.state_dict())
    optB = torch.optim.Adam(netB.parameters(), lr=DL_B_LR)
    ldr = DataLoader(TensorDataset(torch.from_numpy(Xp), torch.from_numpy(Rp)),
                     batch_size=512, shuffle=True)
    netB.train()
    for ep in range(DL_B_EP):
        for xb, yb in ldr:
            optB.zero_grad()
            (((netB(xb) - yb) ** 2).mean()).backward(); optB.step()
    netB.eval()
    Dw = prevê_dlw([di], netB)
    del netB, ldr, Xp, Rp; gc.collect()
    En = wB[0]*Ps + wB[1]*Se + wB[2]*PnFT + wB[3]*Gm + wB[4]*Dw + wB[5]*At
    resB.append({"mae": {m: mae(Y[[di]], p) for m, p in
                           zip(["sazonal", "saz_esc", "lstnet_ft", "lgbm_B", "dlw_B", "amptrend", "ensB"],
                               [Ps, Se, PnFT, Gm, Dw, At, En])},
                 "amp": float(Y[[di]].max() - Y[[di]].min())})
    print(f"dia {dia} pool={len(pool)} ({time.time()-t0:.0f}s): " +
          ", ".join(f"{m}={resB[-1]['mae'][m]:.3f}" for m in ["sazonal", "lgbm_B", "amptrend", "ensB"]), flush=True)

pd.DataFrame([{**{"dia": str((ends[d] + pd.Timedelta(minutes=5)).date())},
               **r["mae"], "amp_dia": r["amp"]} for r, d in zip(resB, daily_idx)]).to_csv(OUT / "metricas_por_dia_walkforward.csv", index=False)
print("walk-forward concluído")

pesos B (subset A renormalizado): {'sazonal': 0.0, 'saz_esc': 0.0, 'lstnet_ft': 0.7197, 'lgbm_B': 0.0, 'dlw_B': 0.0, 'amptrend': 0.2803}


dia 2026-07-12 pool=2606 (17s): sazonal=0.076, lgbm_B=0.150, amptrend=0.190, ensB=0.110


dia 2026-07-13 pool=2894 (35s): sazonal=0.159, lgbm_B=0.249, amptrend=0.136, ensB=0.155


dia 2026-07-14 pool=3182 (53s): sazonal=0.245, lgbm_B=0.307, amptrend=0.272, ensB=0.134


dia 2026-07-15 pool=3470 (73s): sazonal=0.454, lgbm_B=0.595, amptrend=0.970, ensB=0.413


...

dia 2026-07-18 pool=4334 (133s): sazonal=0.059, lgbm_B=0.053, amptrend=0.478, ensB=0.229


dia 2026-07-19 pool=4622 (154s): sazonal=0.062, lgbm_B=0.069, amptrend=0.391, ensB=0.155


dia 2026-07-20 pool=4910 (175s): sazonal=0.088, lgbm_B=0.135, amptrend=0.318, ensB=0.107


dia 2026-07-21 pool=5198 (197s): sazonal=0.219, lgbm_B=0.314, amptrend=0.206, ensB=0.253


walk-forward concluído


## 11. Tabelas finais — A (comparável) + B (walk-forward) + secundária 2594 origens
Primária: holdout diário (10 origens). Secundária: MAE nas 2594 origens do holdout com os modelos do Estágio A (mais potência estatística).

In [12]:
Xda, Yda = X[daily_idx], Y[daily_idx]
ch_da = cheap_preds(Xda)

t0 = time.time()
Ps_da = snaive(Xda); Se_da = saz_escalado(Xda)
Pn02_da = prevê_lstnet(daily_idx, ckpt02["state"])
PnFT_da = prevê_lstnet(daily_idx, ft_state)
GmA_da = prevê_lgbm(daily_idx, models_A)
DwA_da = prevê_dlw(daily_idx, dlw_A)
RgA_da = prevê_ridge(daily_idx)
AtA_da = prevê_amptrend(daily_idx)
EnA_da = ensemble_A(Ps_da, Se_da, Pn02_da, PnFT_da, GmA_da, DwA_da, RgA_da, AtA_da)
print(f"inferência holdout-diário em {time.time()-t0:.0f}s")

linhas = {m: metricas(Yte, p) for m, p in pred_te.items()}
for m, p in [("lstnet02", Pn02_te), ("lstnet_ft", PnFT_te), ("lgbm_A", GmA_te),
              ("dlw_A", DwA_te), ("ridge", RgA_te), ("amptrend", AtA_te), ("ensA", EnA_te)]:
    linhas[m] = metricas(Yte, p)
tab = pd.DataFrame(linhas).T.round(4)
tab.to_csv(OUT / "metricas_baseline.csv")
print("=== teste rolante (Estágio A) ===")
print(tab.to_string())

diarioA = {m: metricas(Yda, ch_da[m]) for m in ["persistencia", "sazonal_naive_288", "media_movel_288"]}
for m, p in [("saz_escalado", Se_da), ("lstnet02", Pn02_da), ("lstnet_ft", PnFT_da),
              ("lgbm_A", GmA_da), ("dlw_A", DwA_da), ("ridge", RgA_da),
              ("amptrend", AtA_da), ("ensA", EnA_da)]:
    diarioA[m] = metricas(Yda, p)
tab_dA = pd.DataFrame(diarioA).T.round(4)
tab_dA.to_csv(OUT / "metricas_holdout.csv")
print("=== holdout diário — Estágio A (comparável) ===")
print(tab_dA.to_string())

# Estágio B: MAE médio dos 10 dias (walk-forward) + baselines p/ referência
Bmae = {m: float(np.mean([r["mae"][m] for r in resB])) for m in resB[0]["mae"]}
print("=== holdout diário — Estágio B walk-forward (MAE) ===")
for m, v in sorted(Bmae.items(), key=lambda kv: kv[1]):
    print(f"{m}: MAE={v:.4f}")

# secundária: 2594 origens do holdout, modelos do Estágio A
Xho, Yho = X[ho], Y[ho]
t0 = time.time()
Ps_ho = snaive(Xho); Se_ho = saz_escalado(Xho)
PnFT_ho = prevê_lstnet(ho, ft_state)
GmA_ho = prevê_lgbm(ho, models_A)
DwA_ho = prevê_dlw(ho, dlw_A)
RgA_ho = prevê_ridge(ho)
AtA_ho = prevê_amptrend(ho)
print(f"inferência holdout-2594 em {time.time()-t0:.0f}s")
sec = {"sazonal_naive_288": metricas(Yho, cheap_preds(Xho)["sazonal_naive_288"]),
       "saz_escalado": metricas(Yho, Se_ho), "lstnet_ft": metricas(Yho, PnFT_ho),
       "lgbm_A": metricas(Yho, GmA_ho), "dlw_A": metricas(Yho, DwA_ho),
       "ridge": metricas(Yho, RgA_ho), "amptrend": metricas(Yho, AtA_ho)}
tab_s = pd.DataFrame(sec).T.round(4)
tab_s.to_csv(OUT / "metricas_holdout_2594.csv")
print("=== holdout 2594 origens (secundária) ===")
print(tab_s.to_string())

por_diaA = pd.DataFrame(
    {"sazonal_naive_288": [mae(Yda[k:k+1], ch_da["sazonal_naive_288"][k:k+1]) for k in range(len(Yda))],
     "saz_escalado": [mae(Yda[k:k+1], Se_da[k:k+1]) for k in range(len(Yda))],
     "lstnet_ft": [mae(Yda[k:k+1], PnFT_da[k:k+1]) for k in range(len(Yda))],
     "lgbm_A": [mae(Yda[k:k+1], GmA_da[k:k+1]) for k in range(len(Yda))],
     "amptrend": [mae(Yda[k:k+1], AtA_da[k:k+1]) for k in range(len(Yda))],
     "ensA": [mae(Yda[k:k+1], EnA_da[k:k+1]) for k in range(len(Yda))],
     "amp_dia": [float(Yda[k].max() - Yda[k].min()) for k in range(len(Yda))]}, 
    index=[str(ends[i].date()) for i in daily_idx])
por_diaA.to_csv(OUT / "metricas_por_dia.csv")
print(por_diaA.round(4).to_string())
print(f"\nRégua 00b (teste): 0.1525 | melhor A: {tab['MAE'].idxmin()} = {tab['MAE'].min():.4f}")
print(f"Régua 00b (holdout): 0.1550 | melhor A: {tab_dA['MAE'].idxmin()} = {tab_dA['MAE'].min():.4f} | melhor B: {min(Bmae, key=Bmae.get)} = {min(Bmae.values()):.4f}")
print(f"pesos A: {pesosA}")

inferência holdout-diário em 0s
=== teste rolante (Estágio A) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.2371  0.3019  3.3575  3.3530
sazonal_naive_288  0.1525  0.1770  2.1668  2.1918
media_movel_288    0.1900  0.2468  2.6526  2.6942
saz_escalado       0.1506  0.1677  2.1472  2.1577
lstnet02           0.0981  0.1198  1.3839  1.3878
lstnet_ft          0.1097  0.1285  1.5586  1.5642
lgbm_A             0.1662  0.2037  2.3609  2.3829
dlw_A              0.1850  0.2153  2.6363  2.6676
ridge              0.1450  0.1641  2.0652  2.0790
amptrend           0.3046  0.3406  4.3144  4.4335
ensA               0.0920  0.1109  1.3042  1.3037
=== holdout diário — Estágio A (comparável) ===
                      MAE    RMSE    MAPE   sMAPE
persistencia       0.4273  0.4921  5.7036  5.6685
sazonal_naive_288  0.1550  0.2233  2.0794  2.1009
media_movel_288    0.4071  0.4916  5.3416  5.4047
saz_escalado       0.1874  0.2406  2.5452  2.5431
lstnet02           0.2428  0.3048  3

inferência holdout-2594 em 4s
=== holdout 2594 origens (secundária) ===
                      MAE    RMSE    MAPE   sMAPE
sazonal_naive_288  0.1509  0.2194  2.0324  2.0591
saz_escalado       0.1788  0.2320  2.4384  2.4430
lstnet_ft          0.1857  0.2391  2.5094  2.4934
lgbm_A             0.1897  0.2497  2.5534  2.5778
dlw_A              0.2046  0.2548  2.7529  2.7656
ridge              0.2182  0.2857  2.8716  2.9110
amptrend           0.4063  0.4666  5.3721  5.5576
            sazonal_naive_288  saz_escalado  lstnet_ft  lgbm_A  amptrend    ensA  amp_dia
2026-07-12             0.0762        0.1097     0.0887  0.1180    0.1902  0.0826     0.77
2026-07-13             0.1595        0.2173     0.1629  0.1775    0.1362  0.2844     0.79
2026-07-14             0.2452        0.2402     0.0979  0.2436    0.2715  0.1870     1.17
2026-07-15             0.4544        0.3979     0.1967  0.4651    0.9695  0.1360     0.97
2026-07-16             0.1219        0.1053     0.2664  0.1768    0.6712  0.17

In [13]:
E = ends[te]
fig, axes = plt.subplots(3, 1, figsize=(12, 9))
for ax, k in zip(axes, [0, len(Xte)//2, -1]):
    tc = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H+2015)), E[k] - pd.Timedelta(minutes=5*H), freq="5min")
    ax.plot(tc, Xte[k][-2016:], lw=0.8, label="contexto (cauda 7d)")
    tf = pd.date_range(E[k] - pd.Timedelta(minutes=5*(H-1)), E[k], freq="5min")
    ax.plot(tf, Yte[k], "k-", lw=1.5, label="real")
    ax.plot(tf, pred_te["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, PnFT_te[k], lw=1, alpha=0.6, label="lstnet_ft")
    ax.plot(tf, GmA_te[k], lw=1, alpha=0.9, label="lgbm_A")
    ax.plot(tf, EnA_te[k], lw=1.2, alpha=0.9, label="ensA")
    ax.set_title(f"origem {E[k]}")
    ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "figs" / "04-forecasts.png")

fig, ax = plt.subplots(figsize=(8, 4))
tab["MAE"].sort_values().plot.barh(ax=ax)
ax.set_title("MAE no teste rolante — Estágio A (menor = melhor)")
fig.tight_layout(); fig.savefig(OUT / "figs" / "05-mae.png")

fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharey=False)
for ax, k in zip(axes.ravel(), range(len(Yda))):
    tf = pd.date_range(ends[daily_idx[k]] - pd.Timedelta(minutes=5*(H-1)), ends[daily_idx[k]], freq="5min")
    ax.plot(tf, Yda[k], "k-", lw=1.2, label="real")
    ax.plot(tf, ch_da["sazonal_naive_288"][k], "--", lw=1, label="sazonal-naive")
    ax.plot(tf, PnFT_da[k], lw=1, alpha=0.6, label="lstnet_ft")
    ax.plot(tf, GmA_da[k], lw=1, alpha=0.9, label="lgbm_A")
    ax.plot(tf, EnA_da[k], lw=1.2, alpha=0.9, label="ensA")
    ax.set_title(f"dia {ends[daily_idx[k]].date()} (MAE ensA={por_diaA['ensA'].iloc[k]:.3f} vs saz={por_diaA['sazonal_naive_288'].iloc[k]:.3f})")
    ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "06-holdout-dias.png")

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].bar(range(len(Yda)), por_diaA["amp_dia"].values)
axes[0].set_title("Amplitude do dia real (max-min, mg/L)")
axes[0].set_xticks(range(len(Yda)), [str(ends[i].date()) for i in daily_idx], rotation=30, fontsize=8)
for m in ["sazonal_naive_288", "saz_escalado", "lstnet_ft", "lgbm_A", "amptrend", "ensA"]:
    axes[1].plot(range(len(Yda)), por_diaA[m].values, marker="o", ms=3, label=m)
axes[1].set_title("MAE por dia — Estágio A (o erro cresce com a amplitude?)")
axes[1].set_xticks(range(len(Yda)), [str(ends[i].date()) for i in daily_idx], rotation=30, fontsize=8)
axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "08-erro-amplitude.png")

wf = pd.read_csv(OUT / "metricas_por_dia_walkforward.csv")
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].bar(range(len(wf)), wf["amp_dia"].values)
axes[0].set_title("Amplitude do dia real (max-min, mg/L)")
axes[0].set_xticks(range(len(wf)), wf["dia"].tolist(), rotation=30, fontsize=8)
for m in ["sazonal", "lgbm_B", "amptrend", "ensB"]:
    axes[1].plot(range(len(wf)), wf[m].values, marker="o", ms=3, label=m)
axes[1].set_title("MAE por dia — Estágio B walk-forward")
axes[1].set_xticks(range(len(wf)), wf["dia"].tolist(), rotation=30, fontsize=8)
axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "figs" / "09-walkforward-dias.png")
print("figs salvas")

figs salvas


## 12. Conclusões e próximos passos

- Réguas do 00b (0,1525 / 0,1550) impressas na §11; o Estágio A é comparável (mesmo protocolo, `te` limpo); o Estágio B é tabela separada de deploy realista.
- O refit em `tr+va` + fine-tune de escala + amplitude-forward testam se o gap era regime de treino — se algum braço vencer no holdout, vira a régua do OD.
- Se nada vencer: o sazonal-naive com H=1d nesta série é um teto duro para modelos univariados; próximos seriam quantis (incerteza) e transferência pós-gap.
- Artefatos em `resultados/06-refit-od/`: `metricas_baseline.csv`, `metricas_holdout.csv`, `metricas_holdout_2594.csv`, `metricas_por_dia.csv`, `metricas_por_dia_walkforward.csv`, `modelos/dlinear_A_od.pt`, `modelos/lstnet_ft_od.pt`, `modelos/ensemble.json`, `modelos/normalizacao.json` e `figs/` (`lgbm_mult_A_steps.pkl` fora do git, regenerável; modelos diários do walk-forward efêmeros).